In [ ]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score


In [ ]:
train_dataset = pd.read_csv("intermediate_data/train_dataset.csv")
train_dataset

In [ ]:
# Same 80/20 split as notebooks 2-4, 6 and 7. The XGBoost model was fit on the 80%,
# so the 20% holdout is what gets explained below.
y = train_dataset.pop('TARGET')
train_dataset.drop(columns=['Unnamed: 0', 'SK_ID_CURR'], inplace=True)
_, X_valid, _, y_valid = train_test_split(train_dataset, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
xgb_pipe = joblib.load("models/xgb_final_pipeline_w_50_optuna.joblib")
# Sanity check: 0.7938, the same holdout AUC as in notebooks 4 and 7.
roc_auc_score(y_valid, xgb_pipe.predict_proba(X_valid)[:, 1])

In [ ]:
[x for x in xgb_pipe.named_steps]

In [ ]:
booster = xgb_pipe.named_steps['xgb_model'].get_booster()
gain = booster.get_score(importance_type="gain")
total_gain = booster.get_score(importance_type="total_gain")
weight = booster.get_score(importance_type="weight")
cover = booster.get_score(importance_type="cover")

In [ ]:
col_names = xgb_pipe.named_steps['preprocess'].get_feature_names_out()
col_names = [col[col.find('__')+2:] if '__' in col else col for col in col_names]

In [ ]:
# get_score() keys features f0, f1, ... by column position and leaves out features never
# used in a split, so look each score up by its key instead of zipping by position.
assert booster.feature_names is None and booster.num_features() == len(col_names)

def importance_df(scores, importance_type):
    return pd.DataFrame({
        "feature": col_names,
        importance_type: [scores.get(f"f{i}", 0.0) for i in range(len(col_names))],
    }).sort_values(importance_type, ascending=False)

gain_df = importance_df(gain, "gain")
total_gain_df = importance_df(total_gain, "total_gain")
weight_df = importance_df(weight, "weight")
cover_df = importance_df(cover, "cover")

print(f"{len(weight)} of {len(col_names)} features are used in at least one split")
total_gain_df.head(20)


In [ ]:
import shap

In [ ]:
# Explain the holdout applicants, which the model did not train on (about 3 minutes on CPU).
X_valid_trans = xgb_pipe.named_steps["preprocess"].transform(X_valid)
feature_names = col_names

model = xgb_pipe.named_steps["xgb_model"]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_valid_trans)


In [ ]:
shap.summary_plot(shap_values, X_valid_trans, feature_names=feature_names)